# Depth Anything V2 — Quick Visualizer

In [ ]:
import os, glob, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import AutoModelForDepthEstimation, AutoImageProcessor
import torch.nn.functional as F

# ── config ──────────────────────────────────────────────────────────────
TEST_DIR   = "data/test"          # folder with .png / .jpg images
SIZE       = "large"              # small | base | large
OUTPUT_RES = 560                  # output depth resolution
N_SHOW     = 6                    # images to display
CMAP       = "inferno"
# ────────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
HF_NAMES = {
    "small": "depth-anything/Depth-Anything-V2-Small-hf",
    "base":  "depth-anything/Depth-Anything-V2-Base-hf",
    "large": "depth-anything/Depth-Anything-V2-Large-hf",
}
processor = AutoImageProcessor.from_pretrained(HF_NAMES[SIZE])
model     = AutoModelForDepthEstimation.from_pretrained(HF_NAMES[SIZE]).eval().to(device)
print(f"Loaded DA2-{SIZE}")

In [ ]:
@torch.no_grad()
def predict(img: Image.Image) -> np.ndarray:
    inputs = processor(images=img, return_tensors="pt").to(device)
    depth  = model(**inputs).predicted_depth          # (1, H', W')
    depth  = F.interpolate(
        depth.unsqueeze(1),
        size=(OUTPUT_RES, OUTPUT_RES),
        mode="bilinear", align_corners=False,
    ).squeeze().cpu().numpy()
    return depth


paths = sorted(
    glob.glob(os.path.join(TEST_DIR, "*.png")) +
    glob.glob(os.path.join(TEST_DIR, "*.jpg"))
)[:N_SHOW]

print(f"{len(paths)} images found")

In [ ]:
cols = 2
rows = len(paths)
fig, axes = plt.subplots(rows, cols, figsize=(10, 4 * rows))
if rows == 1:
    axes = [axes]

for ax_row, path in zip(axes, paths):
    img   = Image.open(path).convert("RGB")
    depth = predict(img)

    ax_row[0].imshow(img)
    ax_row[0].set_title(os.path.basename(path), fontsize=8)
    ax_row[0].axis("off")

    im = ax_row[1].imshow(depth, cmap=CMAP)
    ax_row[1].set_title(f"depth  min={depth.min():.1f}  max={depth.max():.1f}", fontsize=8)
    ax_row[1].axis("off")
    plt.colorbar(im, ax=ax_row[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()